In [1]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

In [2]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Working on device: {device}")

Working on device: mps


In [3]:
#Loading and separating in two columns a small dialog dataset to trained the allready pre-trained model
df = pd.read_csv(
    "dialogs.txt",
    sep="\t",
    header=None,
    names=["Context", "Reply"],
    encoding="utf-8",
    engine="python",
    on_bad_lines="skip"
)

df = df.dropna()
df["Context"] = df["Context"].astype(str).str.strip()
df["Reply"] = df["Reply"].astype(str).str.strip()
df = df[(df["Context"] != "") & (df["Reply"] != "")]

df.head()

,Context,Reply
0,"hi, how are you doing?",i'm fine. how about yourself?
1,i'm fine. how about yourself?,i'm pretty good. thanks for asking.
2,i'm pretty good. thanks for asking.,no problem. so how have you been?
3,no problem. so how have you been?,i've been great. what about you?
4,i've been great. what about you?,i've been good. i'm in school right now.


In [4]:
df["Text"] = "User: " + df["Context"] + "\nAssistant: " + df["Reply"]
df[["Text"]].head()

,Text
0,"User: hi, how are you doing?\nAssistant: i'm f..."
1,User: i'm fine. how about yourself?\nAssistant...
2,User: i'm pretty good. thanks for asking.\nAss...
3,User: no problem. so how have you been?\nAssis...
4,User: i've been great. what about you?\nAssist...


In [5]:
#Upload the preprocesed dataset
dataset=Dataset.from_pandas(df[["Text"]])
dataset

Dataset({
    features: ['Text'],
    num_rows: 3723
})

In [6]:
#Loading the pretrained model from HuggingFace
model_path = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.float32
)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

model = model.to(device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [7]:
#Tokenization
def tokenize_function(example):
    return tokenizer(
        example["Text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print(tokenized_dataset)

Map:   0%|          | 0/3723 [00:00<?, ? examples/s]

Dataset({
    features: ['Text', 'input_ids', 'attention_mask'],
    num_rows: 3723
})


In [8]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  
)

In [9]:
#Training parameters
training_args = TrainingArguments(
    output_dir="./Qwen2.5-1.5B",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=False,
    report_to="none",
    dataloader_pin_memory=False,
)

In [10]:
model.config.use_cache = False
model.gradient_checkpointing_enable()

In [11]:
#Training the model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

In [12]:
trainer.train()

Step,Training Loss
100,2.244979
200,2.131122
300,2.067938
400,1.986758
500,1.848560
600,1.777241
700,1.741094
800,1.665501
900,1.561898
1000,1.121527


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1862, training_loss=1.3911530815312994, metrics={'train_runtime': 3683.3142, 'train_samples_per_second': 2.022, 'train_steps_per_second': 0.506, 'total_flos': 7493219456385024.0, 'train_loss': 1.3911530815312994, 'epoch': 2.0})

In [13]:
#Saving the model
trainer.save_model("./Qwen2.5-1.5B")
tokenizer.save_pretrained("./Qwen2.5-1.5B")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./Qwen2.5-1.5B/tokenizer_config.json',
 './Qwen2.5-1.5B/chat_template.jinja',
 './Qwen2.5-1.5B/tokenizer.json')

In [14]:
#Evalueting
chat_model_path = "./Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(chat_model_path)

model = AutoModelForCausalLM.from_pretrained(
    chat_model_path,
    dtype=torch.float32
)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

model = model.to(device)
model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151645)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), e

In [ ]:
#Chat function
def chat():
    model.eval()
    history = ""

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ["exit", "quit"]:
            break

        history += f"User: {user_input}\nAssistant:"

        inputs = tokenizer(
            history,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=35,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        if "User:" in answer:
            answer = answer.split("User:")[0].strip()

        print("Bot:", answer)
        print()

        history += f" {answer}\n"

chat()

You:  Hello


Bot: what's up? how have you been lately? i haven't heard anything. are you going to be there? if not, someone should set me on fire. that way

